## Loading and testing.

Reload runtime to save memory.

In [2]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00


In [1]:
# if running with colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path='/content/drive/MyDrive/DPO/DPO/'

In [4]:
import gc
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
import pandas as pd
import math
from tqdm import tqdm
import json
import random
import hashlib

BETA = 0.1

BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Store new evaluation results separately from the old 3594 experiment
RESULTS_DIR = Path(path+"eval_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Persistent cache for base/reference-model completion log probabilities.
# Stored under the existing results directory and reused across adapters
# and across later notebook runs.
REFERENCE_CACHE_FILE = RESULTS_DIR / "base_reference_logprobs.jsonl"
REFERENCE_MAX_PROMPT_TOKENS = 512
REFERENCE_MAX_COMPLETION_TOKENS = 256

ADAPTER_COUNTRY_CODES = ["CHN", "JPN", "GBR", "US", "MEX", "ARG", "DEU", 'RUS']
DATA_COUNTRY_CODES = ["CHN", "JPN", "GBR", "US", "MEX", "ARG", "DEU", 'RUS']

# Evaluation selection mode:
#   "all"      -> evaluate every adapter on every country dataset
#   "self"     -> evaluate each adapter only on its matching country dataset
#   "selected" -> use EVALUATION_TARGETS below
EVALUATION_MODE = "self"

# Used only when EVALUATION_MODE == "selected".
# Keys are adapter country codes; values are the country datasets to evaluate on.
# Adapters omitted from this dictionary, or mapped to [], are skipped entirely.
# Example:
# EVALUATION_TARGETS = {
#     "USA": ["USA", "MEX"],
#     "MEX": ["MEX"],
# }
EVALUATION_TARGETS = {
    "USA": ["USA", "MEX"],
    "MEX": ["USA", "MEX"],
}

PATH_COUNTRY_CODES = {
    "USA": "US",
    "MEX": "MEX",
}

DATA_DIR = Path(path)

# New adapter locations
ADAPTER_ROOT = DATA_DIR / "dpo_qlora_adapters"

ADAPTER_DIRS = {
    country: ADAPTER_ROOT / PATH_COUNTRY_CODES.get(country, country)
    for country in ADAPTER_COUNTRY_CODES
}

# New held-out 599-dataset evaluation files
EVAL_FILES = {
    country: DATA_DIR / f"{country}_eval.jsonl"
    for country in DATA_COUNTRY_CODES
}

for country, path in ADAPTER_DIRS.items():
    print(f"{country} adapter: {path}")

for country, path in EVAL_FILES.items():
    print(f"{country} eval: {path}")

CHN adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/CHN
JPN adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/JPN
GBR adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/GBR
US adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/US
MEX adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/MEX
ARG adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/ARG
DEU adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/DEU
RUS adapter: /content/drive/MyDrive/DPO/DPO/dpo_qlora_adapters/RUS
CHN eval: /content/drive/MyDrive/DPO/DPO/CHN_eval.jsonl
JPN eval: /content/drive/MyDrive/DPO/DPO/JPN_eval.jsonl
GBR eval: /content/drive/MyDrive/DPO/DPO/GBR_eval.jsonl
US eval: /content/drive/MyDrive/DPO/DPO/US_eval.jsonl
MEX eval: /content/drive/MyDrive/DPO/DPO/MEX_eval.jsonl
ARG eval: /content/drive/MyDrive/DPO/DPO/ARG_eval.jsonl
DEU eval: /content/drive/MyDrive/DPO/DPO/DEU_eval.jsonl
RUS eval: /content/drive/MyDrive/DPO/DPO/RUS_eval.jsonl


In [5]:
# login with hugging face.
# must have an account with them
# occasionally also requests an access token
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

In [6]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )

In [7]:
def get_model_device(model):
    """
    Returns the device where the model is located.
    Works for regular models and PEFT/QLoRA models.
    """
    return next(model.parameters()).device


@torch.no_grad()
def sequence_logprob(
    model,
    tokenizer,
    prompt_text,
    completion_text,
    max_prompt_tokens=512,
    max_completion_tokens=256,
):
    """
    Computes log p(completion | prompt) for a single prompt-completion pair.

    Important:
    This should use the same prompt formatting as DPO training.
    The log probability is summed over completion tokens only.
    """

    model.eval()
    device = get_model_device(model)

    # Use the same chat-template format used during DPO training.

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    prompt_ids = tokenizer(
        formatted_prompt,
        add_special_tokens=False,
    ).input_ids

    completion_ids = tokenizer(
        completion_text,
        add_special_tokens=False,
    ).input_ids

    # Truncate if needed.
    if len(prompt_ids) > max_prompt_tokens:
        prompt_ids = prompt_ids[-max_prompt_tokens:]

    if len(completion_ids) > max_completion_tokens:
        completion_ids = completion_ids[:max_completion_tokens]

    input_ids = prompt_ids + completion_ids

    # Mask prompt tokens so only completion tokens count in the loss/logprob.
    labels = [-100] * len(prompt_ids) + completion_ids

    input_ids = torch.tensor([input_ids], device=device)
    labels = torch.tensor([labels], device=device)

    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    # Shift for next-token prediction.
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]

    log_probs = torch.log_softmax(shifted_logits, dim=-1)

    mask = shifted_labels.ne(-100)

    safe_labels = shifted_labels.clone()
    safe_labels[~mask] = 0

    token_log_probs = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    completion_logprob = (token_log_probs * mask).sum()

    return float(completion_logprob.detach().cpu())

def _reference_cache_key(
    tokenizer,
    prompt_text,
    completion_text,
    max_prompt_tokens=REFERENCE_MAX_PROMPT_TOKENS,
    max_completion_tokens=REFERENCE_MAX_COMPLETION_TOKENS,
):
    """
    Build a stable key for a base/reference log-probability.

    The key uses the fully formatted prompt, so changing the prompt formatting
    naturally creates a different cache key instead of silently reusing an
    incompatible value.
    """
    formatted_prompt = format_prompt_text(tokenizer, prompt_text)
    payload = {
        "base_model": BASE_MODEL_NAME,
        "formatted_prompt": formatted_prompt,
        "completion": completion_text,
        "max_prompt_tokens": max_prompt_tokens,
        "max_completion_tokens": max_completion_tokens,
    }
    serialized = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def load_reference_logprob_cache(cache_file=REFERENCE_CACHE_FILE):
    """Load any reference log-probabilities saved by earlier runs."""
    cache = {}

    if not cache_file.exists():
        print(f"Reference cache not found yet: {cache_file}")
        return cache

    with open(cache_file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                cache[row["cache_key"]] = float(row["logp"])
            except Exception as exc:
                print(
                    f"Warning: skipped malformed reference-cache line "
                    f"{line_number}: {exc}"
                )

    print(f"Loaded {len(cache)} cached base/reference log-probabilities from {cache_file}")
    return cache


REFERENCE_LOGPROB_CACHE = load_reference_logprob_cache()


def _append_reference_logprob_to_cache(
    cache_key,
    logp,
    prompt_text,
    completion_text,
    cache_file=REFERENCE_CACHE_FILE,
):
    """
    Persist a newly computed reference score immediately.

    Immediate appends make the cache useful even if a long evaluation run is
    interrupted and restarted later.
    """
    row = {
        "cache_key": cache_key,
        "base_model": BASE_MODEL_NAME,
        "max_prompt_tokens": REFERENCE_MAX_PROMPT_TOKENS,
        "max_completion_tokens": REFERENCE_MAX_COMPLETION_TOKENS,
        "prompt": prompt_text,
        "completion": completion_text,
        "logp": float(logp),
    }

    with open(cache_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def get_reference_logprob(
    adapter_model,
    tokenizer,
    prompt_text,
    completion_text,
):
    """
    Return log p(completion | prompt) under the base/reference model.

    A score is computed only when the exact formatted prompt/completion pair
    is absent from the persistent cache. Otherwise the cached value is reused.
    """
    cache_key = _reference_cache_key(
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        completion_text=completion_text,
    )

    if cache_key in REFERENCE_LOGPROB_CACHE:
        return REFERENCE_LOGPROB_CACHE[cache_key]

    # Compute the base/reference score only once. For a PEFT model,
    # disabling the adapter exposes the underlying base model.
    with adapter_model.disable_adapter():
        logp = sequence_logprob(
            adapter_model,
            tokenizer,
            prompt_text,
            completion_text,
            max_prompt_tokens=REFERENCE_MAX_PROMPT_TOKENS,
            max_completion_tokens=REFERENCE_MAX_COMPLETION_TOKENS,
        )

    REFERENCE_LOGPROB_CACHE[cache_key] = logp
    _append_reference_logprob_to_cache(
        cache_key=cache_key,
        logp=logp,
        prompt_text=prompt_text,
        completion_text=completion_text,
    )

    return logp


def dpo_implied_reward_delta(
    adapter_model,
    tokenizer,
    prompt,
    chosen,
    rejected,
    beta=BETA,
):
    """
    Computes the DPO implied reward difference using the same PEFT model.

    Reference/base log-probabilities are cached by formatted prompt-completion
    pair, so each unique base score is computed at most once and is reused
    across adapters, countries, and later notebook runs.
    """

    ref_chosen_logp = get_reference_logprob(
        adapter_model,
        tokenizer,
        prompt,
        chosen,
    )

    ref_rejected_logp = get_reference_logprob(
        adapter_model,
        tokenizer,
        prompt,
        rejected,
    )

    # Adapter model: adapter enabled. These remain adapter-specific and are
    # intentionally recomputed for each adapter being evaluated.
    adapter_chosen_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        chosen,
    )

    adapter_rejected_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        rejected,
    )

    ref_margin = ref_chosen_logp - ref_rejected_logp
    adapter_margin = adapter_chosen_logp - adapter_rejected_logp

    reward_delta = beta * (adapter_margin - ref_margin)

    dpo_pref_prob = 1.0 / (1.0 + math.exp(-reward_delta))

    return {
        "ref_chosen_logp": ref_chosen_logp,
        "ref_rejected_logp": ref_rejected_logp,
        "adapter_chosen_logp": adapter_chosen_logp,
        "adapter_rejected_logp": adapter_rejected_logp,
        "ref_margin": ref_margin,
        "adapter_margin": adapter_margin,
        "dpo_reward_delta": reward_delta,
        "dpo_pref_prob": dpo_pref_prob,
        "dpo_prefers_chosen": reward_delta > 0,
    }

@torch.no_grad()
def generate_model_answer(
    model,
    tokenizer,
    prompt_text,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Generates a free-form answer from the model for a prompt.
    Uses the same prompt format as reward recovery and training.
    """

    model.eval()
    device = get_model_device(model)

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Keep only newly generated tokens, not the prompt.
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated_text

def evaluate_adapter_reward_recovery(
    adapter_model,
    tokenizer,
    eval_file,
    model_name,
    eval_country,
    beta=BETA,
    # CHANGE HERE FOR TEST OUTPUT
    max_examples=200,
    #max_examples=5,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Runs DPO implied reward recovery on a held-out eval JSONL file.

    Also optionally generates a free-form answer from the adapter
    for each prompt and stores it in the result dataframe.
    """

    rows = load_jsonl(eval_file)

    if max_examples is not None:
        rows = rows[:max_examples]

    results = []

    for ex in tqdm(rows, desc=f"Reward recovery: {model_name} on {eval_country}"):
        out = dpo_implied_reward_delta(
            adapter_model=adapter_model,
            tokenizer=tokenizer,
            prompt=ex["prompt"],
            chosen=ex["chosen"],
            rejected=ex["rejected"],
            beta=beta,
        )

        if generate_answers:
            generated_answer = generate_model_answer(
                model=adapter_model,
                tokenizer=tokenizer,
                prompt_text=ex["prompt"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
        else:
            generated_answer = None

        results.append({
            "model": model_name,
            "eval_country": eval_country,
            "country": ex.get("country"),
            "item_id": ex.get("item_id"),
            #"dimension": ex.get("dimension"),
            "gps_dimension": ex.get("gps_dimension"),
            "prompt": ex["prompt"],
            "chosen": ex["chosen"],
            "rejected": ex["rejected"],
            "generated_answer": generated_answer,
            **out,
        })

    return pd.DataFrame(results)



Reference cache not found yet: /content/drive/MyDrive/DPO/DPO/eval_results/base_reference_logprobs.jsonl


In [8]:

def load_base_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    base_model.eval()

    return base_model, tokenizer


def load_adapter_model(adapter_dir):
    base_model, tokenizer = load_base_model_and_tokenizer()

    adapter_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    adapter_model.eval()

    return adapter_model, tokenizer


def cleanup_models(*models):
    for model in models:
        del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [9]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


In [ ]:
# Load one adapter for the smoke test.
# The full cross-evaluation loop below loads and cleans up each adapter one at a time.
#SMOKE_TEST_ADAPTER_COUNTRY = ADAPTER_COUNTRY_CODES[0]
#smoke_model, tokenizer = load_adapter_model(ADAPTER_DIRS[SMOKE_TEST_ADAPTER_COUNTRY])


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [10]:
def smoke_generate_once(model, tokenizer, prompt, max_new_tokens=80):
    was_training = model.training
    model.eval()

    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ]
    terminators = [
        t for t in terminators
        if t is not None and t != tokenizer.unk_token_id
    ]

    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=terminators,
            use_cache=True,
        )

    prompt_len = encoded["input_ids"].shape[-1]
    generated_ids = output_ids[0][prompt_len:]

    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("\n[smoke test]")
    print(text)

    if "gibberish_score" in globals():
        print("gibberish_score:", gibberish_score(text))

    if was_training:
        model.train()

    return text

In [ ]:
"""
smoke_generate_once(
    smoke_model,
    tokenizer,
    "Explain in one paragraph why housing affordability matters.\n\nAnswer:"
)
"""

In [ ]:
# ============================
# Cross-evaluate adapters
# ============================

all_adapter_dfs = []

if EVALUATION_MODE not in {"all", "self", "selected"}:
    raise ValueError(
        f"Invalid EVALUATION_MODE={EVALUATION_MODE!r}. "
        "Use 'all', 'self', or 'selected'."
    )

if EVALUATION_MODE == "selected":
    unknown_adapters = set(EVALUATION_TARGETS) - set(ADAPTER_COUNTRY_CODES)
    if unknown_adapters:
        raise ValueError(
            f"EVALUATION_TARGETS contains adapters not listed in "
            f"ADAPTER_COUNTRY_CODES: {sorted(unknown_adapters)}"
        )

    unknown_datasets = {
        country
        for targets in EVALUATION_TARGETS.values()
        for country in targets
        if country not in EVAL_FILES
    }
    if unknown_datasets:
        raise ValueError(
            f"EVALUATION_TARGETS contains datasets not listed in "
            f"DATA_COUNTRY_CODES: {sorted(unknown_datasets)}"
        )

for adapter_country in ADAPTER_COUNTRY_CODES:
    if EVALUATION_MODE == "all":
        eval_countries = list(DATA_COUNTRY_CODES)
    elif EVALUATION_MODE == "self":
        eval_countries = [adapter_country] if adapter_country in EVAL_FILES else []
    else:
        eval_countries = list(EVALUATION_TARGETS.get(adapter_country, []))

    if not eval_countries:
        print(f"Skipping {adapter_country}: no evaluation datasets selected.")
        continue

    adapter_dir = ADAPTER_DIRS[adapter_country]
    adapter_model, tokenizer = load_adapter_model(adapter_dir)

    adapter_eval_dfs = []

    for eval_country in eval_countries:
        eval_file = EVAL_FILES[eval_country]

        df_adapter_on_eval = evaluate_adapter_reward_recovery(
            adapter_model=adapter_model,
            tokenizer=tokenizer,
            eval_file=eval_file,
            model_name=f"{adapter_country}_adapter",
            eval_country=eval_country,
            beta=BETA,
            generate_answers=True,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
        )

        df_adapter_on_eval.to_csv(
            RESULTS_DIR / f"reward_recovery_{adapter_country}_adapter_on_{eval_country}.csv",
            index=False,
        )

        adapter_eval_dfs.append(df_adapter_on_eval)
        all_adapter_dfs.append(df_adapter_on_eval)

    if adapter_eval_dfs:
        df_adapter = pd.concat(
            adapter_eval_dfs,
            ignore_index=True,
        )

        df_adapter.to_csv(
            RESULTS_DIR / f"reward_recovery_{adapter_country}_adapter.csv",
            index=False,
        )

    cleanup_models(adapter_model)
    del adapter_model, tokenizer

if all_adapter_dfs:
    df_all_adapters = pd.concat(
        all_adapter_dfs,
        ignore_index=True,
    )

    df_all_adapters.to_csv(
        RESULTS_DIR / "reward_recovery_all_adapters.csv",
        index=False,
    )

    df_all_adapters.head()


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Reward recovery: CHN_adapter on CHN:   3%|▎         | 4/132 [00:43<22:38, 10.62s/it]